# Dense Mapped Logit-KD: Production ONNX FP32 Export & Parity

Notebook này bắt đầu nhánh triển khai sau khi phương pháp và Held-out Test đã
được khóa. Checkpoint triển khai là **seed 123**, được chọn bằng CIDEr-D trên
Validation (`1.0819028987`) trước khi Test được mở; tuyệt đối không chọn seed
theo kết quả Test.

Mục tiêu:

1. xác minh chuỗi khóa phương pháp → Held-out Test → comparator completion;
2. nạp checkpoint seed 123 bằng `strict=True`;
3. xuất encoder và decoder FP32 sang ONNX;
4. kiểm tra cấu trúc ONNX và parity với PyTorch;
5. đóng gói vocabulary, contract và SHA-256 cho bước INT8 tiếp theo.

## Kaggle inputs bắt buộc — chỉ 5 input

1. `Dense_Mapped_LogitKD_Held_Out_Test_Static_Input`;
2. output `XLKD05_Dense_Mapped_LogitKD_123_Full_Run`;
3. output `XLKD05_Dense_Mapped_LogitKD_Held_Out_Test`;
4. output `XLKD05_Dense_Mapped_LogitKD_Held_Out_Comparator_Completion`.
5. dataset `ONNX_Runtime_Python313_Offline`.

Không gắn MS COCO, Teacher, Baseline, seed 42/2026 hoặc các notebook
quantization. Notebook chỉ dùng input tổng hợp để đo parity kỹ thuật; không đọc
Validation và không mở lại Test.

## Runtime

- Accelerator: `None / CPU`.
- Internet: `Off`.
- Cần có sẵn `torch`, `torchvision`, `onnx`, `onnxruntime`.

## Cell 1 — Authorization, selection policy và integrity

In [1]:
import csv
import hashlib
import importlib
import importlib.util
import json
import os
import platform
import shutil
import statistics
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
assert INPUT_ROOT.is_dir(), "Notebook phải chạy trên Kaggle."

RUN_NAME = "XLKD05_Dense_Mapped_LogitKD_production_onnx_fp32"
OUT = WORK_ROOT / RUN_NAME
AUDITS = OUT / "audits"
CONFIGS = OUT / "configs"
EXPORTS = OUT / "exports"
PACKAGE = OUT / "package"
RESULTS = OUT / "results"
MANIFESTS = OUT / "manifests"
for directory in (OUT, AUDITS, CONFIGS, EXPORTS, PACKAGE, RESULTS, MANIFESTS):
    directory.mkdir(parents=True, exist_ok=True)

def now():
    return datetime.now(timezone.utc).isoformat()

def sha256(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)

def candidates(name):
    return sorted(INPUT_ROOT.rglob(name), key=lambda path: str(path))

def exactly_one(name):
    matches = [path for path in candidates(name) if path.is_file()]
    assert len(matches) == 1, (
        f"Cần đúng 1 file {name!r}; tìm thấy {len(matches)}: {matches[:10]}"
    )
    return matches[0]

def by_name_and_hash(name, expected_sha256):
    matches = [
        path for path in candidates(name)
        if path.is_file() and sha256(path) == expected_sha256
    ]
    assert matches, f"Không tìm thấy {name!r} có SHA-256 {expected_sha256}."
    return matches[0]

def resolve_reference(marker_path, reference):
    direct = marker_path.parent / reference["relative_path"]
    if direct.is_file() and sha256(direct) == reference["sha256"]:
        return direct
    return by_name_and_hash(Path(reference["relative_path"]).name, reference["sha256"])

def verify_manifest(root, manifest_path, expected_rows=None):
    with Path(manifest_path).open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    if expected_rows is not None:
        assert len(rows) == int(expected_rows), (len(rows), expected_rows)
    for row in rows:
        target = (Path(root) / row["relative_path"]).resolve()
        assert target.is_file(), target
        expected_size = row.get("bytes", row.get("size_bytes"))
        assert target.stat().st_size == int(expected_size), target
        assert sha256(target) == row["sha256"], target
    return rows

LOCK_MARKER_PATH = exactly_one(
    "XLKD05_Dense_Mapped_LogitKD_FINAL_METHOD_LOCKED.json"
)
TEST_COMPLETE_PATH = exactly_one(
    "XLKD05_Dense_Mapped_LogitKD_HELD_OUT_TEST_COMPLETE.json"
)
TEST_OPENED_PATH = exactly_one(
    "XLKD05_Dense_Mapped_LogitKD_TEST_OPENED.json"
)
COMPARATOR_COMPLETE_PATH = exactly_one(
    "XLKD05_Dense_Mapped_LogitKD_HELD_OUT_TEST_COMPARATORS_COMPLETE.json"
)

lock_marker = read_json(LOCK_MARKER_PATH)
test_complete = read_json(TEST_COMPLETE_PATH)
test_opened = read_json(TEST_OPENED_PATH)
comparator_complete = read_json(COMPARATOR_COMPLETE_PATH)

assert lock_marker["status"] == "locked"
assert test_complete["status"] == "complete"
assert comparator_complete["status"] == "complete"
assert test_complete["test_open_count"] == 1
assert comparator_complete["test_open_count"] == 1
assert comparator_complete["new_test_opening"] is False
assert comparator_complete["continuation_of_opening_id"] == test_opened["opening_id"]
assert comparator_complete["checkpoint_or_seed_selection_on_test"] is False
assert comparator_complete["threshold_tuning_on_test"] is False
assert comparator_complete["posthoc_method_changes"] is False

CONTRACT_PATH = resolve_reference(LOCK_MARKER_PATH, lock_marker["contract"])
contract = read_json(CONTRACT_PATH)
assert contract["status"] == "locked"
assert contract["method_id"] == "DENSE_MAPPED_LOGIT_KD"
assert contract["selection_primary"] == "Validation CIDEr-D"
assert contract["posthoc_method_changes_allowed"] is False

# Chọn duy nhất bằng Validation, không đọc metric Test để chọn checkpoint.
validation_ranked = sorted(
    contract["checkpoints"],
    key=lambda row: (-float(row["best_validation_cider_d"]), int(row["seed"])),
)
selected = validation_ranked[0]
assert int(selected["seed"]) == 123
assert abs(float(selected["best_validation_cider_d"]) - 1.081902898658348) < 1e-12
CHECKPOINT_PATH = by_name_and_hash(
    Path(selected["relative_to_run"]).name, selected["sha256"]
)

VOCAB_SHA256 = "1c6c7e6146d2700932c82be6f076d93153c757c10a1844076abaea9c11496f97"
DECODER_CONTRACT_SHA256 = "46655293c4a64506ea842dd0389715fbf64d19ae389c3ba79d2997fe5c3d62dc"
VOCAB_PATH = by_name_and_hash("word_vocabulary_minfreq5.json", VOCAB_SHA256)
DECODER_CONTRACT_PATH = by_name_and_hash(
    "shared_transformer_decoder_contract.json", DECODER_CONTRACT_SHA256
)

comparator_manifest_path = resolve_reference(
    COMPARATOR_COMPLETE_PATH, comparator_complete["manifest"]
)
comparator_rows = verify_manifest(
    COMPARATOR_COMPLETE_PATH.parent,
    comparator_manifest_path,
    comparator_complete["manifest"]["rows"],
)

# Kaggle Python 3.13 có thể có ONNX nhưng thiếu ONNX Runtime. Khi thiếu, chỉ
# cài wheel Linux cp313 đã gắn dưới dạng input; không truy cập Internet và không
# thay NumPy của image Kaggle.
ort_installed_from_offline_wheel = False
if importlib.util.find_spec("onnxruntime") is None:
    ort_wheels = sorted(INPUT_ROOT.rglob("onnxruntime-*-cp313-cp313-manylinux*.whl"))
    assert len(ort_wheels) == 1, (
        "Thiếu đúng một wheel onnxruntime Linux cp313. Hãy gắn input "
        "ONNX_Runtime_Python313_Offline. Tìm thấy: " + str(ort_wheels)
    )
    wheel_dir = ort_wheels[0].parent
    support_wheels = []
    for pattern in ("flatbuffers-*.whl", "packaging-*.whl", "protobuf-*.whl"):
        matches = sorted(wheel_dir.glob(pattern))
        assert len(matches) == 1, (pattern, matches)
        support_wheels.append(matches[0])
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-index",
        "--no-deps",
        *[str(path) for path in support_wheels],
        str(ort_wheels[0]),
    ])
    importlib.invalidate_caches()
    ort_installed_from_offline_wheel = True

runtime_modules = {
    name: importlib.util.find_spec(name) is not None
    for name in ("torch", "torchvision", "onnx", "onnxruntime", "numpy")
}
assert all(runtime_modules.values()), runtime_modules

preflight = {
    "schema_version": "dense_mapped_logit_kd_production_onnx_preflight_v1",
    "created_at_utc": now(),
    "status": "passed",
    "method": contract["method"],
    "selection_policy": "highest locked Validation CIDEr-D; never Test",
    "selected_seed": 123,
    "selected_validation_cider_d": float(selected["best_validation_cider_d"]),
    "selected_checkpoint": {
        "path": str(CHECKPOINT_PATH),
        "bytes": CHECKPOINT_PATH.stat().st_size,
        "sha256": sha256(CHECKPOINT_PATH),
    },
    "test_chain": {
        "opening_id": test_opened["opening_id"],
        "open_count": 1,
        "comparator_manifest_rows_verified": len(comparator_rows),
        "new_test_opening": False,
        "test_data_read_by_this_notebook": False,
    },
    "runtime_modules": runtime_modules,
    "onnxruntime_installed_from_offline_wheel": ort_installed_from_offline_wheel,
    "environment": {"python": sys.version, "platform": platform.platform()},
}
write_json(AUDITS / "cell01_preflight.json", preflight)

print("Preflight                : PASSED")
print("Selected checkpoint      : seed 123")
print("Selection source         : Validation CIDEr-D only")
print("Checkpoint SHA-256       :", selected["sha256"])
print("Comparator manifest      :", len(comparator_rows), "/", len(comparator_rows))
print("Held-out Test reopened   : False")

Processing /kaggle/input/datasets/nguynthnhdan/onnx-runtime-python313-offline/flatbuffers-25.12.19-py2.py3-none-any.whl
Processing /kaggle/input/datasets/nguynthnhdan/onnx-runtime-python313-offline/packaging-26.3-py3-none-any.whl
Processing /kaggle/input/datasets/nguynthnhdan/onnx-runtime-python313-offline/protobuf-7.36.2-py3-none-any.whl
Processing /kaggle/input/datasets/nguynthnhdan/onnx-runtime-python313-offline/onnxruntime-1.28.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
flatbuffers is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
packaging is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
Preflight                : PASSED
Selected checkpoint      : se

## Cell 2 — Strict checkpoint load và fixed mobile contract

In [2]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import mobilenet_v3_large

torch.set_grad_enabled(False)
torch.manual_seed(123)
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

decoder_config = read_json(DECODER_CONTRACT_PATH)["architecture"]
assert int(decoder_config["vocabulary_size"]) == 10080
assert int(decoder_config["maximum_decoder_steps"]) == 31

class S1Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = mobilenet_v3_large(weights=None).features
        self.projection = nn.Linear(960, 256)
        self.visual_position_embedding = nn.Parameter(torch.empty(1, 49, 256))
        self.output_norm = nn.LayerNorm(256)

    def forward(self, images):
        feature = self.backbone(images)
        tokens = feature.flatten(2).transpose(1, 2).contiguous()
        return self.output_norm(
            self.projection(tokens) + self.visual_position_embedding
        )

class SharedDecoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.vocab = int(config["vocabulary_size"])
        self.d_model = int(config["d_model"])
        self.maximum_steps = int(config["maximum_decoder_steps"])
        self.token_embedding = nn.Embedding(self.vocab, self.d_model, padding_idx=0)
        self.position_embedding = nn.Embedding(self.maximum_steps, self.d_model)
        self.embedding_dropout = nn.Dropout(float(config["dropout"]))
        layer = nn.TransformerDecoderLayer(
            d_model=self.d_model,
            nhead=int(config["nhead"]),
            dim_feedforward=int(config["dim_feedforward"]),
            dropout=float(config["dropout"]),
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer_decoder = nn.TransformerDecoder(
            layer, num_layers=int(config["num_layers"])
        )
        self.final_norm = nn.LayerNorm(self.d_model)
        self.output_bias = nn.Parameter(torch.zeros(self.vocab))

    def forward(self, ids, memory, mask):
        length = ids.shape[1]
        positions = torch.arange(length, device=ids.device).unsqueeze(0).expand(
            ids.shape[0], -1
        )
        target = (
            self.token_embedding(ids) * math.sqrt(self.d_model)
            + self.position_embedding(positions)
        )
        target = self.embedding_dropout(target)
        causal = torch.triu(
            torch.ones(length, length, dtype=torch.bool, device=ids.device),
            diagonal=1,
        )
        hidden = self.transformer_decoder(
            target,
            memory,
            tgt_mask=causal,
            tgt_key_padding_mask=~mask.bool(),
            # The causal mask above is fixed and exact. Supplying the hint
            # prevents PyTorch ONNX export from entering the expensive
            # data-dependent causal-mask detection path.
            tgt_is_causal=True,
        )
        hidden = self.final_norm(hidden)
        return F.linear(hidden, self.token_embedding.weight, self.output_bias)

class ExportSafeDecoder(nn.Module):
    """ONNX-safe evaluation decomposition of SharedDecoder."""
    def __init__(self, source):
        super().__init__()
        self.vocab = source.vocab
        self.d_model = source.d_model
        self.maximum_steps = source.maximum_steps
        self.token_embedding = source.token_embedding
        self.position_embedding = source.position_embedding
        self.layers = source.transformer_decoder.layers
        self.final_norm = source.final_norm
        self.output_bias = source.output_bias
        self.register_buffer(
            "causal_mask",
            torch.triu(torch.ones(self.maximum_steps, self.maximum_steps, dtype=torch.bool), diagonal=1),
            persistent=False,
        )

    def _attention(self, module, query, key, value, valid_keys, causal):
        width = self.d_model
        heads = module.num_heads
        head_width = width // heads
        weight = module.in_proj_weight
        bias = module.in_proj_bias
        q = F.linear(query, weight[:width], bias[:width])
        k = F.linear(key, weight[width:2 * width], bias[width:2 * width])
        v = F.linear(value, weight[2 * width:], bias[2 * width:])
        batch = query.shape[0]
        query_length = query.shape[1]
        key_length = key.shape[1]
        q = q.reshape(batch, query_length, heads, head_width).transpose(1, 2)
        k = k.reshape(batch, key_length, heads, head_width).transpose(1, 2)
        v = v.reshape(batch, key_length, heads, head_width).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(head_width)
        if causal:
            scores = scores.masked_fill(
                self.causal_mask[:query_length, :key_length].unsqueeze(0).unsqueeze(0),
                -1.0e4,
            )
        if valid_keys is not None:
            scores = scores.masked_fill(~valid_keys.bool().unsqueeze(1).unsqueeze(1), -1.0e4)
        probabilities = torch.softmax(scores, dim=-1)
        context = torch.matmul(probabilities, v)
        context = context.transpose(1, 2).contiguous().reshape(batch, query_length, width)
        return F.linear(context, module.out_proj.weight, module.out_proj.bias)

    def forward(self, ids, memory, mask):
        positions = torch.arange(self.maximum_steps, device=ids.device).unsqueeze(0).expand(ids.shape[0], -1)
        hidden = (
            self.token_embedding(ids) * math.sqrt(self.d_model)
            + self.position_embedding(positions)
        )
        for layer in self.layers:
            normalized = layer.norm1(hidden)
            hidden = hidden + self._attention(
                layer.self_attn, normalized, normalized, normalized, mask, True
            )
            normalized = layer.norm2(hidden)
            hidden = hidden + self._attention(
                layer.multihead_attn, normalized, memory, memory, None, False
            )
            normalized = layer.norm3(hidden)
            hidden = hidden + F.linear(
                F.gelu(F.linear(normalized, layer.linear1.weight, layer.linear1.bias)),
                layer.linear2.weight,
                layer.linear2.bias,
            )
        hidden = self.final_norm(hidden)
        return F.linear(hidden, self.token_embedding.weight, self.output_bias)

encoder = S1Encoder().cpu().eval()
decoder = SharedDecoder(decoder_config).cpu().eval()
payload = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
assert payload["schema_version"] in {
    "xlkd04_dense_mapped_checkpoint_v1",
    "xlkd05_dense_mapped_checkpoint_v1",
}
assert payload["experiment_id"] == "XLKD_EXPANSIONNET_S1_DENSE_MAPPED_LOGITKD"
assert int(payload["seed"]) == 123
assert payload["test_set_status"] == "sealed"
assert payload["test_set_used"] is False
encoder_result = encoder.load_state_dict(payload["student_encoder_state"], strict=True)
decoder_result = decoder.load_state_dict(payload["student_decoder_state"], strict=True)
assert not encoder_result.missing_keys and not encoder_result.unexpected_keys
assert not decoder_result.missing_keys and not decoder_result.unexpected_keys
del payload

ENCODER_PARAMETERS = sum(parameter.numel() for parameter in encoder.parameters())
DECODER_PARAMETERS = sum(parameter.numel() for parameter in decoder.parameters())
assert ENCODER_PARAMETERS == 3231024
assert DECODER_PARAMETERS == 5759328

IMAGE = torch.linspace(-2.0, 2.0, 1 * 3 * 224 * 224).reshape(1, 3, 224, 224)
TOKEN_IDS = torch.zeros((3, 31), dtype=torch.long)
TOKEN_MASK = torch.zeros((3, 31), dtype=torch.int64)
prefixes = ([1], [1, 20, 31, 17, 2], [1] + list(range(4, 34)))
for row, prefix in enumerate(prefixes):
    usable = prefix[:31]
    TOKEN_IDS[row, :len(usable)] = torch.tensor(usable, dtype=torch.long)
    TOKEN_MASK[row, :len(usable)] = 1

with torch.inference_mode():
    MEMORY = encoder(IMAGE)
    DECODER_MEMORY = MEMORY.expand(3, -1, -1).contiguous()
    LOGITS = decoder(TOKEN_IDS, DECODER_MEMORY, TOKEN_MASK)
    export_decoder = ExportSafeDecoder(decoder).cpu().eval()
    EXPORT_LOGITS = export_decoder(TOKEN_IDS, DECODER_MEMORY, TOKEN_MASK)
assert tuple(MEMORY.shape) == (1, 49, 256)
assert tuple(LOGITS.shape) == (3, 31, 10080)
assert torch.isfinite(MEMORY).all() and torch.isfinite(LOGITS).all()
EXPORT_WRAPPER_MAX_ABS_ERROR = float((LOGITS - EXPORT_LOGITS).abs().max().item())
assert EXPORT_WRAPPER_MAX_ABS_ERROR <= 2e-4, EXPORT_WRAPPER_MAX_ABS_ERROR
assert torch.equal(LOGITS.argmax(dim=-1), EXPORT_LOGITS.argmax(dim=-1))

load_audit = {
    "schema_version": "dense_mapped_logit_kd_strict_load_v1",
    "created_at_utc": now(),
    "status": "passed",
    "seed": 123,
    "checkpoint_sha256": sha256(CHECKPOINT_PATH),
    "strict_encoder_load": True,
    "strict_decoder_load": True,
    "export_wrapper_equivalence_max_abs_error": EXPORT_WRAPPER_MAX_ABS_ERROR,
    "encoder_parameters": ENCODER_PARAMETERS,
    "decoder_parameters": DECODER_PARAMETERS,
    "total_parameters": ENCODER_PARAMETERS + DECODER_PARAMETERS,
    "estimated_fp32_parameter_mib": (
        (ENCODER_PARAMETERS + DECODER_PARAMETERS) * 4 / 2**20
    ),
    "fixed_contract": {
        "image": [1, 3, 224, 224],
        "visual_memory": [1, 49, 256],
        "decoder_input_ids": [3, 31],
        "decoder_visual_memory": [3, 49, 256],
        "decoder_attention_mask": [3, 31],
        "logits": [3, 31, 10080],
    },
}
write_json(AUDITS / "cell02_strict_load.json", load_audit)

print("Strict checkpoint load   : PASSED")
print("Encoder parameters       :", f"{ENCODER_PARAMETERS:,}")
print("Decoder parameters       :", f"{DECODER_PARAMETERS:,}")
print("Total parameters         :", f"{ENCODER_PARAMETERS + DECODER_PARAMETERS:,}")
print("FP32 parameter estimate  :", f"{load_audit['estimated_fp32_parameter_mib']:.2f} MiB")
print("Export wrapper error      :", f"{EXPORT_WRAPPER_MAX_ABS_ERROR:.6e}")

Strict checkpoint load   : PASSED
Encoder parameters       : 3,231,024
Decoder parameters       : 5,759,328
Total parameters         : 8,990,352
FP32 parameter estimate  : 34.30 MiB
Export wrapper error      : 2.479553e-05


## Cell 3 — ONNX FP32 export và graph validation

In [3]:
import numpy as np
import onnx
import onnxruntime as ort

OPSET = 18
ENCODER_ONNX = EXPORTS / "dense_mapped_logit_kd_s1_seed123_encoder_fp32.onnx"
DECODER_ONNX = EXPORTS / "dense_mapped_logit_kd_s1_seed123_decoder_fp32.onnx"

def export_onnx(model, args, path, input_names, output_names):
    attempts = []
    if path.exists():
        path.unlink()
    print(f"[ONNX] Starting export: {path.name}", flush=True)
    # The deployment contract is deliberately fixed-shape. PyTorch's
    # dynamo exporter can remain indefinitely in symbolic causal-mask
    # analysis for nn.TransformerDecoder on the Kaggle Python 3.13 image.
    # Use the deterministic TorchScript exporter directly; ORT parity below
    # remains the semantic acceptance gate.
    torch.onnx.export(
        model,
        args,
        str(path),
        input_names=input_names,
        output_names=output_names,
        opset_version=OPSET,
        export_params=True,
        do_constant_folding=True,
        dynamo=False,
    )
    mode = "legacy_fixed_shape_direct"
    print(f"[ONNX] Export finished: {path.name} ({mode})", flush=True)
    assert path.is_file() and path.stat().st_size > 1_000_000
    print(f"[ONNX] Loading and checking: {path.name}", flush=True)
    graph = onnx.load(str(path), load_external_data=True)
    onnx.checker.check_model(graph, full_check=True)
    print(f"[ONNX] Checker passed: {path.name}", flush=True)
    return {
        "export_mode": mode,
        "failed_attempts": attempts,
        "bytes": path.stat().st_size,
        "sha256": sha256(path),
        "opset": max(item.version for item in graph.opset_import if item.domain in {"", "ai.onnx"}),
        "nodes": len(graph.graph.node),
        "initializers": len(graph.graph.initializer),
    }

started = time.perf_counter()
print("[ONNX] Exporter revision: primitive_attention_r2", flush=True)
encoder_export = export_onnx(
    encoder,
    (IMAGE,),
    ENCODER_ONNX,
    ["image"],
    ["visual_memory"],
)
decoder_export = export_onnx(
    export_decoder,
    (TOKEN_IDS, DECODER_MEMORY, TOKEN_MASK),
    DECODER_ONNX,
    ["decoder_input_ids", "visual_memory", "decoder_attention_mask"],
    ["logits"],
)
decoder_export["wrapper"] = "primitive_attention_mathematically_equivalent"
decoder_export["wrapper_equivalence_max_abs_error"] = EXPORT_WRAPPER_MAX_ABS_ERROR
export_seconds = time.perf_counter() - started

export_audit = {
    "schema_version": "dense_mapped_logit_kd_onnx_fp32_export_v1",
    "created_at_utc": now(),
    "status": "passed",
    "opset": OPSET,
    "encoder": encoder_export,
    "decoder": decoder_export,
    "fixed_shape_mobile_contract": True,
    "beam_size": 3,
    "sequence_buffer": 31,
    "export_seconds": export_seconds,
    "training_performed": False,
    "validation_data_read": False,
    "test_data_read": False,
}
write_json(AUDITS / "cell03_onnx_export.json", export_audit)

print("ONNX export              : PASSED")
print("Encoder mode             :", encoder_export["export_mode"])
print("Decoder mode             :", decoder_export["export_mode"])
print("Encoder size             :", f"{encoder_export['bytes'] / 2**20:.2f} MiB")
print("Decoder size             :", f"{decoder_export['bytes'] / 2**20:.2f} MiB")
print("ONNX checker             : PASSED 2 / 2")

[ONNX] Exporter revision: primitive_attention_r2
[ONNX] Starting export: dense_mapped_logit_kd_s1_seed123_encoder_fp32.onnx


/tmp/ipykernel_16/931574586.py:19: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


[ONNX] Export finished: dense_mapped_logit_kd_s1_seed123_encoder_fp32.onnx (legacy_fixed_shape_direct)
[ONNX] Loading and checking: dense_mapped_logit_kd_s1_seed123_encoder_fp32.onnx
[ONNX] Checker passed: dense_mapped_logit_kd_s1_seed123_encoder_fp32.onnx
[ONNX] Starting export: dense_mapped_logit_kd_s1_seed123_decoder_fp32.onnx
[ONNX] Export finished: dense_mapped_logit_kd_s1_seed123_decoder_fp32.onnx (legacy_fixed_shape_direct)
[ONNX] Loading and checking: dense_mapped_logit_kd_s1_seed123_decoder_fp32.onnx
[ONNX] Checker passed: dense_mapped_logit_kd_s1_seed123_decoder_fp32.onnx
ONNX export              : PASSED
Encoder mode             : legacy_fixed_shape_direct
Decoder mode             : legacy_fixed_shape_direct
Encoder size             : 12.32 MiB
Decoder size             : 31.84 MiB
ONNX checker             : PASSED 2 / 2


## Cell 4 — PyTorch ↔ ONNX Runtime parity và CPU reference latency

In [4]:
PROVIDERS = ["CPUExecutionProvider"]
session_options = ort.SessionOptions()
session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
encoder_session = ort.InferenceSession(
    str(ENCODER_ONNX), sess_options=session_options, providers=PROVIDERS
)
decoder_session = ort.InferenceSession(
    str(DECODER_ONNX), sess_options=session_options, providers=PROVIDERS
)

ENCODER_ATOL = 1e-4
DECODER_ATOL = 2e-3
cases = []
for case_seed in (7, 42, 123, 2026, 9001):
    generator = torch.Generator().manual_seed(case_seed)
    image = torch.randn((1, 3, 224, 224), generator=generator)
    ids = torch.randint(4, 10080, (3, 31), generator=generator, dtype=torch.long)
    ids[:, 0] = 1
    mask = torch.ones((3, 31), dtype=torch.int64)
    lengths = (1, 11, 31)
    for row, length in enumerate(lengths):
        ids[row, length:] = 0
        mask[row, length:] = 0

    with torch.inference_mode():
        pt_memory = encoder(image)
        pt_decoder_memory = pt_memory.expand(3, -1, -1).contiguous()
        pt_logits = decoder(ids, pt_decoder_memory, mask)
    ort_memory = encoder_session.run(None, {"image": image.numpy()})[0]
    ort_decoder_memory = np.repeat(ort_memory, 3, axis=0)
    ort_logits = decoder_session.run(
        None,
        {
            "decoder_input_ids": ids.numpy(),
            "visual_memory": ort_decoder_memory,
            "decoder_attention_mask": mask.numpy(),
        },
    )[0]

    encoder_error = float(np.max(np.abs(pt_memory.numpy() - ort_memory)))
    decoder_error = float(np.max(np.abs(pt_logits.numpy() - ort_logits)))
    valid_positions = [(row, length - 1) for row, length in enumerate(lengths)]
    agreements = []
    for row, position in valid_positions:
        pt_top1 = int(pt_logits[row, position].argmax().item())
        ort_top1 = int(np.argmax(ort_logits[row, position]))
        agreements.append(pt_top1 == ort_top1)
    cases.append({
        "case_seed": case_seed,
        "encoder_max_abs_error": encoder_error,
        "decoder_max_abs_error": decoder_error,
        "valid_position_top1_agreement": sum(agreements) / len(agreements),
    })

max_encoder_error = max(row["encoder_max_abs_error"] for row in cases)
max_decoder_error = max(row["decoder_max_abs_error"] for row in cases)
top1_agreement = statistics.fmean(
    row["valid_position_top1_agreement"] for row in cases
)
assert max_encoder_error <= ENCODER_ATOL, max_encoder_error
assert max_decoder_error <= DECODER_ATOL, max_decoder_error
assert top1_agreement == 1.0, top1_agreement

def percentile(values, q):
    return float(np.percentile(np.asarray(values, dtype=np.float64), q))

latency_image = np.zeros((1, 3, 224, 224), dtype=np.float32)
latency_ids = TOKEN_IDS.numpy()
latency_mask = TOKEN_MASK.numpy()
for _ in range(3):
    warm_memory = encoder_session.run(None, {"image": latency_image})[0]
    decoder_session.run(None, {
        "decoder_input_ids": latency_ids,
        "visual_memory": np.repeat(warm_memory, 3, axis=0),
        "decoder_attention_mask": latency_mask,
    })

encoder_ms, decoder_ms = [], []
for _ in range(20):
    start = time.perf_counter()
    memory_np = encoder_session.run(None, {"image": latency_image})[0]
    encoder_ms.append((time.perf_counter() - start) * 1000)
    start = time.perf_counter()
    decoder_session.run(None, {
        "decoder_input_ids": latency_ids,
        "visual_memory": np.repeat(memory_np, 3, axis=0),
        "decoder_attention_mask": latency_mask,
    })
    decoder_ms.append((time.perf_counter() - start) * 1000)

parity = {
    "schema_version": "dense_mapped_logit_kd_onnx_fp32_parity_v1",
    "created_at_utc": now(),
    "status": "passed",
    "providers": encoder_session.get_providers(),
    "cases": cases,
    "thresholds": {
        "encoder_max_abs_error": ENCODER_ATOL,
        "decoder_max_abs_error": DECODER_ATOL,
        "valid_position_top1_agreement": 1.0,
    },
    "observed": {
        "encoder_max_abs_error": max_encoder_error,
        "decoder_max_abs_error": max_decoder_error,
        "valid_position_top1_agreement": top1_agreement,
    },
    "cpu_reference_latency_ms": {
        "runs": 20,
        "encoder_p50": percentile(encoder_ms, 50),
        "encoder_p90": percentile(encoder_ms, 90),
        "encoder_p95": percentile(encoder_ms, 95),
        "decoder_single_call_p50": percentile(decoder_ms, 50),
        "decoder_single_call_p90": percentile(decoder_ms, 90),
        "decoder_single_call_p95": percentile(decoder_ms, 95),
        "environment_only_not_android_claim": True,
    },
    "validation_data_read": False,
    "test_data_read": False,
}
write_json(RESULTS / "onnx_fp32_parity_and_cpu_reference.json", parity)

print("FP32 parity              : PASSED")
print("Encoder max abs error    :", f"{max_encoder_error:.6e}")
print("Decoder max abs error    :", f"{max_decoder_error:.6e}")
print("Valid-position top-1     :", f"{top1_agreement:.1%}")
print("CPU numbers              : reference only; not Android benchmark")

FP32 parity              : PASSED
Encoder max abs error    : 1.347065e-05
Decoder max abs error    : 4.768372e-05
Valid-position top-1     : 100.0%
CPU numbers              : reference only; not Android benchmark


## Cell 5 — Deployment package, manifest và authorization INT8

In [5]:
PACKAGE_ENCODER = PACKAGE / "dense_mapped_logit_kd_s1_seed123_encoder_fp32.onnx"
PACKAGE_DECODER = PACKAGE / "dense_mapped_logit_kd_s1_seed123_decoder_fp32.onnx"
PACKAGE_VOCAB = PACKAGE / "word_vocabulary_minfreq5.json"
PACKAGE_DECODER_CONTRACT = PACKAGE / "shared_transformer_decoder_contract.json"
shutil.copy2(ENCODER_ONNX, PACKAGE_ENCODER)
shutil.copy2(DECODER_ONNX, PACKAGE_DECODER)
shutil.copy2(VOCAB_PATH, PACKAGE_VOCAB)
shutil.copy2(DECODER_CONTRACT_PATH, PACKAGE_DECODER_CONTRACT)

deployment_contract = {
    "schema_version": "dense_mapped_logit_kd_s1_seed123_mobile_contract_v1",
    "created_at_utc": now(),
    "method": contract["method"],
    "student": "S1 MobileNetV3-Large",
    "seed": 123,
    "selection": {
        "source": "locked Validation CIDEr-D",
        "value": float(selected["best_validation_cider_d"]),
        "selected_on_test": False,
    },
    "runtime": "ONNX Runtime",
    "precision": "FP32",
    "opset": OPSET,
    "preprocessing": {
        "resize_shortest_side": 256,
        "resize_interpolation": "bicubic",
        "center_crop": [224, 224],
        "layout": "NCHW",
        "dtype": "float32",
        "mean": [0.485, 0.456, 0.406],
        "std": [0.229, 0.224, 0.225],
    },
    "encoder": {
        "file": PACKAGE_ENCODER.name,
        "input": {"image": [1, 3, 224, 224], "dtype": "float32"},
        "output": {"visual_memory": [1, 49, 256], "dtype": "float32"},
    },
    "decoder": {
        "file": PACKAGE_DECODER.name,
        "export_graph": "primitive_attention_mathematically_equivalent",
        "parity_reference": "original_checkpoint_loaded_nn_transformer_decoder",
        "wrapper_equivalence_max_abs_error": EXPORT_WRAPPER_MAX_ABS_ERROR,
        "inputs": {
            "decoder_input_ids": {"shape": [3, 31], "dtype": "int64"},
            "visual_memory": {"shape": [3, 49, 256], "dtype": "float32"},
            "decoder_attention_mask": {"shape": [3, 31], "dtype": "int64"},
        },
        "output": {"logits": [3, 31, 10080], "dtype": "float32"},
    },
    "generation": {
        "beam_size": 3,
        "length_penalty": 0.7,
        "minimum_lexical_tokens": 2,
        "maximum_new_tokens": 31,
        "pad_id": 0,
        "bos_id": 1,
        "eos_id": 2,
        "unk_id": 3,
        "forbidden_generation_token_ids": [0, 1, 3],
    },
    "artifacts": {},
}
for label, path in {
    "encoder": PACKAGE_ENCODER,
    "decoder": PACKAGE_DECODER,
    "vocabulary": PACKAGE_VOCAB,
    "decoder_contract": PACKAGE_DECODER_CONTRACT,
}.items():
    deployment_contract["artifacts"][label] = {
        "file": path.name,
        "bytes": path.stat().st_size,
        "sha256": sha256(path),
    }
DEPLOYMENT_CONTRACT_PATH = PACKAGE / "deployment_contract.json"
write_json(DEPLOYMENT_CONTRACT_PATH, deployment_contract)

authorization = {
    "schema_version": "dense_mapped_logit_kd_int8_validation_authorized_v1",
    "created_at_utc": now(),
    "status": "authorized",
    "required_next": "Validation-only INT8 calibration, parity and quality gate",
    "source_precision": "FP32",
    "selected_seed": 123,
    "constraints": {
        "new_training": False,
        "checkpoint_reselection": False,
        "test_opening": False,
        "calibration_split": "validation_only",
        "calibration_image_ids_must_be_frozen_before_quantization": True,
        "compare_int8_against_fp32_on_same_validation_images": True,
        "android_benchmark_only_after_quality_gate": True,
    },
}
write_json(OUT / "INT8_VALIDATION_GATE_AUTHORIZED.json", authorization)

artifact_paths = []
for directory in (AUDITS, CONFIGS, EXPORTS, PACKAGE, RESULTS):
    artifact_paths.extend(path for path in directory.rglob("*") if path.is_file())
artifact_paths.append(OUT / "INT8_VALIDATION_GATE_AUTHORIZED.json")

manifest_path = MANIFESTS / "production_onnx_fp32_manifest.csv"
with manifest_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["relative_path", "bytes", "sha256"])
    writer.writeheader()
    for path in sorted(artifact_paths, key=lambda item: item.relative_to(OUT).as_posix()):
        writer.writerow({
            "relative_path": path.relative_to(OUT).as_posix(),
            "bytes": path.stat().st_size,
            "sha256": sha256(path),
        })

completion = {
    "schema_version": "dense_mapped_logit_kd_production_onnx_fp32_complete_v1",
    "created_at_utc": now(),
    "status": "complete",
    "decision": "fp32_onnx_parity_passed_int8_validation_gate_authorized",
    "method": contract["method"],
    "selected_seed": 123,
    "selection_source": "locked Validation CIDEr-D only",
    "onnx_exports": "2 / 2",
    "onnx_checker": "2 / 2 passed",
    "fp32_parity": "passed",
    "decoder_export_graph": "primitive_attention_mathematically_equivalent",
    "decoder_wrapper_equivalence_max_abs_error": EXPORT_WRAPPER_MAX_ABS_ERROR,
    "encoder_max_abs_error": max_encoder_error,
    "decoder_max_abs_error": max_decoder_error,
    "valid_position_top1_agreement": top1_agreement,
    "package_contract": {
        "relative_path": DEPLOYMENT_CONTRACT_PATH.relative_to(OUT).as_posix(),
        "sha256": sha256(DEPLOYMENT_CONTRACT_PATH),
    },
    "manifest": {
        "relative_path": manifest_path.relative_to(OUT).as_posix(),
        "sha256": sha256(manifest_path),
        "rows": len(artifact_paths),
    },
    "training_performed": False,
    "validation_data_read": False,
    "test_data_read": False,
    "new_test_opening": False,
    "test_open_count": 1,
}
COMPLETE_MARKER = OUT / "XLKD05_Dense_Mapped_LogitKD_PRODUCTION_ONNX_FP32_COMPLETE.json"
write_json(COMPLETE_MARKER, completion)

print("Production ONNX FP32     : COMPLETE")
print("Selected seed            : 123 (Validation-only selection)")
print("ONNX exports             : 2 / 2")
print("FP32 parity              : PASSED")
print("Held-out Test reopened   : False")
print("Next                     : INT8 Validation gate")
print("Complete marker          :", COMPLETE_MARKER)

Production ONNX FP32     : COMPLETE
Selected seed            : 123 (Validation-only selection)
ONNX exports             : 2 / 2
FP32 parity              : PASSED
Held-out Test reopened   : False
Next                     : INT8 Validation gate
Complete marker          : /kaggle/working/XLKD05_Dense_Mapped_LogitKD_production_onnx_fp32/XLKD05_Dense_Mapped_LogitKD_PRODUCTION_ONNX_FP32_COMPLETE.json
